# Panel de jueces LLM (A2)

Pre-etiquetado de `data/dataset/instances.jsonl` con tres jueces (Llama 3.1 8B, Qwen 2.5 7B y Mistral 7B v0.3) servidos con vLLM en una GPU A100.

Antes de ejecutar:

- Subir `EvidenCite_A2.zip` a `MyDrive/EvidenCite_A2/`.
- Aceptar en Hugging Face las licencias de `meta-llama/Llama-3.1-8B-Instruct` y `mistralai/Mistral-7B-Instruct-v0.3`, y guardar el token como secreto `HF_TOKEN` en Colab.
- Entorno de ejecución con GPU A100.

Con `SAMPLE = 60` se hace la prueba corta. Para la corrida completa se cambia a `SAMPLE = None` y se ejecutan de nuevo esa celda y las siguientes. Los votos quedan en caché en Drive, así que si la sesión se cae se puede retomar.

Librerías necesarias: vllm, openai, pydantic, pyyaml, scikit-learn.

In [ ]:
!pip install -q vllm openai pydantic pyyaml scikit-learn

In [3]:
import json, os, shutil, signal, subprocess, time, urllib.request, zipfile
from importlib.metadata import version
from pathlib import Path
import yaml
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/EvidenCite_A2')
REPO = Path('/content/EvidenCite')
if not REPO.exists():
    with zipfile.ZipFile(DRIVE / 'EvidenCite_A2.zip') as z:
        z.extractall(REPO)
os.chdir(REPO)
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
for name in ['labeled', 'logs']:
    if (DRIVE / name).exists():
        shutil.copytree(DRIVE / name, REPO / 'data' / name, dirs_exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('vllm', version('vllm'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NVIDIA A100-SXM4-40GB, 40960 MiB

vllm 0.33.0


In [9]:
CONFIG = 'configs/labeling.yaml'
JUDGES = {j['id']: j['model_name'] for j in yaml.safe_load(open(CONFIG))['judges']}
SAMPLE = None
JUDGES

{'llama31_8b': 'meta-llama/Llama-3.1-8B-Instruct',
 'qwen25_7b': 'Qwen/Qwen2.5-7B-Instruct',
 'mistral_7b': 'mistralai/Mistral-7B-Instruct-v0.3'}

In [10]:
def start_server(model):
    log = open(f"/content/vllm_{model.split('/')[-1]}.log", 'w')
    proc = subprocess.Popen(
        ['vllm', 'serve', model, '--port', '8000', '--max-model-len', '4096', '--gpu-memory-utilization', '0.90'],
        stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
    )
    t0 = time.time()
    while time.time() - t0 < 1800:
        if proc.poll() is not None:
            raise RuntimeError(f'vLLM se detuvo, revisar {log.name}')
        try:
            with urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5) as r:
                if r.status == 200:
                    print(model, f'listo en {time.time() - t0:.0f}s')
                    return proc
        except Exception:
            pass
        time.sleep(10)
    stop_server(proc)
    raise TimeoutError(model)


def stop_server(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(timeout=120)
    except ProcessLookupError:
        pass
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
    time.sleep(15)


def save_to_drive():
    for name in ['labeled', 'logs']:
        shutil.copytree(REPO / 'data' / name, DRIVE / name, dirs_exist_ok=True)


def run_panel(*extra):
    args = ['python', '-m', 'src.labeling.judge_panel', '--config', CONFIG, *extra]
    if SAMPLE:
        args += ['--sample', str(SAMPLE)]
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end='')
    if p.returncode:
        raise RuntimeError(f'judge_panel termino con codigo {p.returncode}')

In [11]:
for judge_id, model in JUDGES.items():
    t0 = time.time()
    proc = start_server(model)
    try:
        run_panel('--judges', judge_id)
    finally:
        stop_server(proc)
        save_to_drive()
    print(judge_id, f'{(time.time() - t0) / 60:.1f} min')

meta-llama/Llama-3.1-8B-Instruct listo en 120s
2026-11-06 03:00:38,639 [INFO] src.labeling.judge_panel: Cargando instancias desde data/dataset/instances.jsonl ...
2026-11-06 03:00:39,631 [INFO] src.labeling.judges: Juez [llama31_8b]: 60 votos recuperados desde caché
2026-11-06 03:00:39,713 [INFO] src.labeling.judge_panel: Iniciando votación con juez [llama31_8b] (meta-llama/Llama-3.1-8B-Instruct) ...
2026-11-06 03:00:39,723 [INFO] src.labeling.judges: Juez [llama31_8b]: 18658 instancias (60 en caché, 18598 por consultar)
2026-11-06 03:00:55,462 [INFO] src.labeling.judges: Juez [llama31_8b]: 200/18658 (1.1%) en 15.7s
2026-11-06 03:01:05,421 [INFO] src.labeling.judges: Juez [llama31_8b]: 400/18658 (2.1%) en 25.7s
2026-11-06 03:01:15,563 [INFO] src.labeling.judges: Juez [llama31_8b]: 600/18658 (3.2%) en 35.8s
2026-11-06 03:01:25,495 [INFO] src.labeling.judges: Juez [llama31_8b]: 800/18658 (4.3%) en 45.8s
2026-11-06 03:01:35,591 [INFO] src.labeling.judges: Juez [llama31_8b]: 1000/18658 (5.

In [12]:
run_panel()
save_to_drive()

2026-11-06 03:47:10,269 [INFO] src.labeling.judge_panel: Cargando instancias desde data/dataset/instances.jsonl ...
2026-11-06 03:47:11,703 [INFO] src.labeling.judges: Juez [llama31_8b]: 18658 votos recuperados desde caché
2026-11-06 03:47:11,953 [INFO] src.labeling.judges: Juez [qwen25_7b]: 18658 votos recuperados desde caché
2026-11-06 03:47:12,170 [INFO] src.labeling.judges: Juez [mistral_7b]: 18641 votos recuperados desde caché
2026-11-06 03:47:12,212 [INFO] src.labeling.judge_panel: Iniciando votación con juez [llama31_8b] (meta-llama/Llama-3.1-8B-Instruct) ...
2026-11-06 03:47:12,223 [INFO] src.labeling.judges: Juez [llama31_8b]: 18658 instancias (18658 en caché, 0 por consultar)
2026-11-06 03:47:12,703 [INFO] src.labeling.judges: Juez [llama31_8b]: 200/18658 (1.1%) en 0.5s
2026-11-06 03:47:12,705 [INFO] src.labeling.judges: Juez [llama31_8b]: 400/18658 (2.1%) en 0.5s
2026-11-06 03:47:12,706 [INFO] src.labeling.judges: Juez [llama31_8b]: 600/18658 (3.2%) en 0.5s
2026-11-06 03:47:

In [13]:
m = json.loads((REPO / 'data' / 'labeled' / 'labeling_metrics.json').read_text())
for k in ['total_instances', 'splits', 'label_source_distribution', 'decision_cases', 'pairwise_cohen_kappa', 'mean_pairwise_kappa']:
    print(k, m[k])
g = m['gold_calibration']
print('gold', g['evaluated_instances'], g['panel_accuracy'], g['per_judge_accuracy'])
for j, u in m['judge_usage'].items():
    print(j, 'votos', u['votes'], 'abstenciones', u['abstentions'], 'latencia_ms', round(u['mean_latency_ms']))

total_instances 18658
splits {'train': 13077, 'test': 2798, 'dev': 2783}
label_source_distribution {'human_gold': 3629, 'needs_review': 7637, 'llm_panel': 7392}
decision_cases {'A': 4864, 'B': 2528, 'C': 4692, 'D': 2945, 'F': 3629}
pairwise_cohen_kappa {'llama31_8b__vs__mistral_7b': 0.36634167632329495, 'llama31_8b__vs__qwen25_7b': 0.4327939861193667, 'mistral_7b__vs__qwen25_7b': 0.39871726084414105}
mean_pairwise_kappa 0.3992843077622676
gold 3629 0.38660788095894183 {'llama31_8b': 0.3799944888399008, 'mistral_7b': 0.32717241379310347, 'qwen25_7b': 0.35409203637365666}
llama31_8b votos 18658 abstenciones 0 latencia_ms 1571
qwen25_7b votos 18658 abstenciones 0 latencia_ms 1333
mistral_7b votos 18658 abstenciones 17 latencia_ms 1445
